In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


In [3]:
print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER GLOBAL - FASE 3 (SALING SILANG & AUTO-SKIP) 🚀 ")
print("================================================================================")

# ================================================================================
# TAHAP 1: MEMUAT SE LURUH BERKAS PICKLE DAN MERGE JADI SATU PINTU
# ================================================================================
all_fase_3_data = {}

# 1. Load File Cimut
try:
    with open('fase_3_cimut.pkl', 'rb') as f:
        all_fase_3_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Cimut.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_3_afrida.pkl', 'rb') as f:
        all_fase_3_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Afrida.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif (Jika ada file terpisah, opsional)
try:
    if os.path.exists('fase_3_hanif.pkl'):
        with open('fase_3_hanif.pkl', 'rb') as f:
            all_fase_3_data.update(pickle.load(f))
        print("✓ Berhasil memuat data hasil konversi Hanif.")
except:
    pass

 🚀 SETUP INSERT HANDLER GLOBAL - FASE 3 (SALING SILANG & AUTO-SKIP) 🚀 
✓ Berhasil memuat data hasil konversi Cimut.
⚠️ Gagal memuat file pkl Afrida: (<StringDtype(storage='python', na_value=nan)>, array(['Kelas', 'HR / GA', 'test'], dtype=object))
✓ Berhasil memuat data hasil konversi Hanif.


In [4]:
# ================================================================================
# TAHAP 2: ATUR URUTAN STRATEGIS SALING SILANG (CIMUT, AFRIDA, & HANIF)
# ================================================================================
# Susunan di bawah ini diatur ketat lintas personel agar Foreign Key aman masuk ke MySQL!
tables_to_insert_ordered = [
    # --- BLOK A: PENDAFTARAN & SDM (Karya Hanif) ---
    'pelamar',                  # Induk data pelamar kerja/kursus
    'pelamar_kerja',            # Detail pelamar posisi kerja
    'pelamar_sekolah',          # Riwayat sekolah pelamar
    'pelamar_kursus',           # Riwayat kursus pelamar
    'progres_pelamar',          # Log catatan tahapan seleksi
    'rekrutmen_pelamar',        # Keputusan akhir rekrutmen pelamar
    'pengajuan_karyawan',       # Form pengajuan penambahan staff baru
    'histori_pengajuan',        # Log alur persetujuan pengajuan staff

    # --- BLOK B: SURAT-MENYURAT & SOP (Karya Afrida) ---
    'sop',                      # Standar operasional prosedur instansi
    'surat_keluar',             # Log keluar dokumen/surat resmi
    'verifikasi_surat_keluar',  # Log persetujuan surat keluar oleh atasan
    'surat_tugas',              # Surat perintah penugasan formal
    'surat_tugas_anggota',      # Anggota staff yang terikat di dalam surat tugas

    # --- BLOK C: MARKETING & ADMISI CALON SISWA (Karya Cimut) ---
    'kontak_prospek',           # Database mentah leads / prospek marketing
    'calon_siswa',              # Formulir profil utama calon siswa baru
    'calon_siswa_ortu',         # Data wali / orang tua calon siswa
    'calon_siswa_akademik',     # Riwayat background akademik calon siswa
    'calon_siswa_bayar',        # Log transaksi pembayaran formulir/DP awal
    'calon_siswa_jadwal',       # Plotting jadwal tes/interview calon siswa
    'calon_siswa_kursus',       # Pilihan program kursus yang diminati calon siswa
    'calon_siswa_proses',       # Jalur perkembangan dokumen admisinya
    'calon_siswa_status_logs',  # Log perubahan status akhir (Diterima/Ditolak/Pending)

    # --- BLOK D: LOGISTIK & OPERASIONAL INTERNAL (Karya Cimut) ---
    'pengadaan',                # Form pengajuan belanja/pengadaan aset barang
    'peminjaman',               # Log pinjam pakai sarana prasarana oleh staff
    'problem'                   # Log laporan kerusakan/kendala teknis fasilitas
]

In [8]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT DENGAN RINGKASAN DI ATAS & DIAGNOSTIK ERROR DI BAWAH
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DI BELAKANG LAYAR
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {
                'status': 'not_found', 
                'rows': 0, 
                'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl'
            }
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {
                'status': 'empty', 
                'rows': 0, 
                'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)'
            }
            continue
            
        try:
            # Bersihkan kolom kosong murni agar tidak merusak placeholder query
            df_to_push = df_target.dropna(axis=1, how='all')
            
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            
            # Gunakan INSERT IGNORE untuk auto-skip duplikat primary key
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            # Konversi DataFrame ke Native List Python (Hancurkan tipe data NumPy)
            raw_numpy_list = df_to_push.to_numpy().tolist()
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN"] else x for x in row) 
                for row in raw_numpy_list
            ]
            
            # Eksekusi massal
            cursor.executemany(insert_query, clean_data_tuples)
            db_connection.commit()
            
            total_rows = len(clean_data_tuples)
            results[table_name] = {
                'status': 'success', 
                'rows': total_rows, 
                'msg': f'✓ {table_name}: Sukses diproses! Sebanyak {total_rows} baris sukses dimasukkan / di-skip aman.'
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'rows': 0, 
                'msg': f'✗ {table_name}: Gagal total saat insert - Alasan: {e}'
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    # 1. Cetak yang sukses dulu biar rapi
    print("🟢 TABEL YANG SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] == 'success':
            print(f"  {results[table_name]['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses)")

    print("\n🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):")
    failed_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] in ['failed', 'not_found', 'empty']:
            print(f"  {results[table_name]['msg']}")
            failed_exist = True
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada yang gagal.")
            
    print("================================================================================\n")


    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            # Taktik A: Jika Sukses, tampilkan preview standard 5 baris teratas
            if results[table_name]['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name].head(5)) 
                print("-" * 80)
                
            # Taktik B: Jika GAGAL, tembak dan kuliti struktur datanya secara transparan!
            elif results[table_name]['status'] == 'failed':
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Alasan MySQL Menolak: {results[table_name]['msg']}")
                print("-" * 50)
                print("Berikut 5 baris sampel data yang gagal dikirim, cek tipe datanya (apakah ada .0 atau string aneh):")
                display(tables_data[table_name].head(5))
                print(f"\nTipe data kolom internal DataFrame untuk tabel '{table_name}':")
                # Menampilkan tipe data internal pandas agar ketahuan mana float ghaib / objek aneh
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

In [9]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL (MENGGUNAKAN VERSI RINGKASAN ATAS)
# ================================================================================
results_fase_3 = insert_data_with_preview_and_skip_v2(
    db_connection=db_new, 
    cursor=cursor_new, 
    tables_data=all_fase_3_data, 
    ordered_list=tables_to_insert_ordered
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)

 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG SUKSES MASUK:
  ✓ pelamar_kerja: Sukses diproses! Sebanyak 67 baris sukses dimasukkan / di-skip aman.
  ✓ pelamar_sekolah: Sukses diproses! Sebanyak 53 baris sukses dimasukkan / di-skip aman.
  ✓ pelamar_kursus: Sukses diproses! Sebanyak 50 baris sukses dimasukkan / di-skip aman.
  ✓ progres_pelamar: Sukses diproses! Sebanyak 403 baris sukses dimasukkan / di-skip aman.
  ✓ rekrutmen_pelamar: Sukses diproses! Sebanyak 281 baris sukses dimasukkan / di-skip aman.
  ✓ pengajuan_karyawan: Sukses diproses! Sebanyak 33 baris sukses dimasukkan / di-skip aman.
  ✓ histori_pengajuan: Sukses diproses! Sebanyak 79 baris sukses dimasukkan / di-skip aman.
  ✓ kontak_prospek: Sukses diproses! Sebanyak 214 baris sukses dimasukkan / di-skip aman.
  ✓ calon_siswa: Sukses diproses! Sebanyak 214 baris sukses dimasukkan / di-skip aman.
  ✓ calon_siswa_ortu: Suks

,id_pelamar,id_pengajuan,email_pelamar,nama_lengkap,nama_panggilan,jenis_kelamin,tempat_lahir,tanggal_lahir,alamat_ktp,alamat_domisili,...,penggunaan_laptop,skor_toefl,ekspektasi_gaji,tautan_berkas,alasan_resign,skor_iq,foto_iq,foto_minat,foto_kepribadian,created_at
0,649ab1b2c5a8f20230627165354,NaN,ditari@leapsurabaya.sch.id,None,None,None,None,None,None,None,...,Tidak Pernah,None,0,None,None,None,None,None,None,NaT
1,649e4885684b120230630101413,NaN,hartikaharahap95@gmail.com,Hartika Prawidaningrum Harahap,Tika,Perempuan,Sidoarjo,NaT,Perum Grand Surya Cluster Jupiter Blok D12/16 ...,Perum Grand Surya Cluster Jupiter Blok D12/16 ...,...,Pernah,507,4550000,https://drive.google.com/open?id=1Z_FpilagwmNd...,sedang tidak bekerja,100,1688095776_3de967eefe836d28e873.jpeg,1688095993_b75d248ce60436d0d4a1.jpg,1688096006_bf7b1c0e082c6aaf5e67.jpeg,2023-06-29 10:24:53
2,64a4ddd6bea4320230705100454,NaN,admin@gmail.com,sdasd,sadas,Perempuan,asd,None,asd,asda,...,Tidak Pernah,asd,0,None,None,None,None,None,None,NaT
3,64a4f88430f6b20230705115844,NaN,nirmalapradnyas@gmail.com,Ni Putu Jayanti Nirmala Pradnya Santosa,Nirmala,Perempuan,Surabaya,NaT,Bendul Merisi Permai blok C no 22 Surabaya,Surabaya,...,Tidak Pernah,517,0,None,None,None,None,None,None,NaT
4,64a52e5a7896920230705154826,8.0,nirmalapradnyas@gmail.com,Ni Putu Jayanti Nirmala Pradnya Santosa,Nirmala,Perempuan,Surabaya,NaT,Bendul Merisi Permai blok C no 22 Surabaya,Surabaya,...,Pernah,517,0,https://drive.google.com/file/d/1u8_ZrepEz7mEG...,-,94,1688617417_0765b3a992194874930b.png,1688617538_60b8cc4bd0dd8da5c0b4.jpg,1688617547_8eb3154d5d4cb7210165.png,2023-07-05 11:17:01



Tipe data kolom internal DataFrame untuk tabel 'pelamar':
id_pelamar                     object
id_pengajuan                  float64
email_pelamar                  object
nama_lengkap                   object
nama_panggilan                 object
jenis_kelamin                  object
tempat_lahir                   object
tanggal_lahir                  object
alamat_ktp                     object
alamat_domisili                object
nomor_wa                       object
akun_linkedin                  object
akun_instagram                 object
akun_facebook                  object
sosmed_lain                    object
spesifikasi_laptop             object
internet                       object
kegiatan_sekarang              object
rencana_karir                  object
mobilitas                      object
sumber_info                    object
siap_wfo                       object
tanggal_bergabung              object
kategori_pelamar               object
riwayat_kerja                

,id_pelamar_kerja,id_pelamar,nama_perusahaan,periode,jabatan,deskripsi_kerja
0,3,U00003,Coding Bee Academy,2021-2022,Educator,<p>- Membuat lesson plan</p>\r\n<p>- Membuat s...
1,4,U00019,Pusat Bahasa UINSA Surabaya,2011 - sampai sekarang,Tutor Bahasa Inggris,<p>Mengajar dua kelas pada semester 1 dan 2. D...
2,5,U00026,PT Aku Pintar Indonesia,2019-2021,Tutor Team Lead dan English Tutor,"<p><span style=""color: rgba(0, 0, 0, 0.9); fon..."
3,6,U00011,INFOMEDIA NUSANTARA,2018-2020,CALL CENTER BNI,<p>Melayani keluhan dan kebutuhan pelanggan BN...
4,7,U00033,LKP LEAP English & Digital Surabaya,2022-Sekarang,Part time pengajar Bahasa Inggris,<p>Mengajar Siswa</p>


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PELAMAR_SEKOLAH]
--------------------------------------------------


...

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PEMINJAMAN]
--------------------------------------------------


,id_pinjam,tanggal_pinjam,keperluan,id_user,status_pinjam,catatan_sarpras,created_at
0,2,2023-06-11,<p>pinjam kamera - fun class tk mitra - 1 - 11...,U00026,Diajukan,None,2023-06-12 13:20:39
1,3,2023-06-11,<p>1. kamera - fun class TK mitra - 1 - 11 Jun...,U00026,Diajukan,None,2023-06-12 13:21:35
2,5,2023-08-21,<p>1. Tablet Leap - 1 - Jaga-jaga untuk Mid Te...,U00026,Diajukan,None,2023-08-16 15:01:55
3,6,2023-08-23,<p>Pinjam kamera untuk rekaman video checklist...,U00033,Diajukan,None,2023-08-23 10:19:12
4,7,2023-10-11,<p>1. Tablet Leap - 1 - Jaga-jaga untuk Final ...,U00026,Diajukan,None,2023-10-10 15:30:53


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: PROBLEM]
--------------------------------------------------


,id_problem,detail_masalah,id_user,status_perbaikan,tanggal_lapor,tanggal_selesai,catatan_teknisi,gambar_problem
0,43,AC Kelas Miss Erika kurang dingin ( belakang,U00012,Diajukan,2023-06-28 16:07:52,NaT,"sudah info ke Pak Irawan,\r\nTukang AC masih l...",None
1,58,Boya/mic untuk kelas hybrid tidak berfungsi (,U00026,Diajukan,2023-07-06 10:45:03,2023-07-17,None,None
2,60,tegangan listrik di ruang kelas belakang dapur...,U00033,Diajukan,2023-07-13 16:59:57,2023-08-10,pemberian stabilizer,None
3,61,ac brisik,U00033,Diajukan,2023-07-14 09:24:29,2023-07-25,sudah tidak berisik,None
4,62,Kabel power monitor PC room 2 longgar. Saat me...,U00036,Diajukan,2023-07-17 15:04:01,2023-07-17,None,None


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [7]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 3 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_3 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )